In [2]:
"""
================================================================================
PROJECT: Stats NZ Labour Market Data Pipeline (Fixed Star Schema ETL)
FILE NAME: 01_etl_labour_market_pipeline.py
DESCRIPTION:
  1. CLEANUP: Forcibly removes old database file to guarantee clean execution.
  2. EXTRACT: Reads Stats NZ CSV datasets (skipping population rebase duplicates).
  3. TRANSFORM: Cleans numeric & text columns, parses period dates, handles nulls.
  4. DIMENSIONAL MODEL: Explicit DDL with PRIMARY & FOREIGN KEY constraints.
  5. POWER BI OPTIMIZATION: Fixes Year/Quarter data types for seamless ODBC import.
  6. VIEWS & AUDIT: Re-builds analytical SQL views and runs star schema audits.
================================================================================
"""

import os
import sqlite3
import pandas as pd
import numpy as np

# ------------------------------------------------------------------------------
# STEP 1: CONFIGURATION & PATH SETUP
# ------------------------------------------------------------------------------
BASE_DIR = r"D:\STATS NZ DATASET"
OUTPUT_DB_PATH = r"D:\ProjectData\LabourMarket_Gold_DW.db"

# Excluded HLFS_POP (rebase file) to avoid double-counting HLFS records
CSV_FILES = {
    "MEI": os.path.join(BASE_DIR, r"employment-indicators-july-2026\employment-indicators-july-2026-csv-tables.csv"),
    "HLFS": os.path.join(BASE_DIR, r"labour-market-statistics-june-2026\hlfs-jun26qtr-csv.csv"),
    "LCI": os.path.join(BASE_DIR, r"labour-market-statistics-june-2026\lci-jun26qtr-csv.csv"),
    "LMS": os.path.join(BASE_DIR, r"labour-market-statistics-june-2026\lms-jun26qtr-tables.csv"),
    "QES": os.path.join(BASE_DIR, r"labour-market-statistics-june-2026\qes-jun26qtr-csv.csv")
}

# Forcibly delete existing database file to wipe stale data completely
if os.path.exists(OUTPUT_DB_PATH):
    try:
        os.remove(OUTPUT_DB_PATH)
        print(f"🧹 Removed existing database at {OUTPUT_DB_PATH}")
    except PermissionError:
        print(f"⚠️ Warning: Database file is locked. Please close Power BI or SQLite connections.")

os.makedirs(os.path.dirname(OUTPUT_DB_PATH), exist_ok=True)
conn = sqlite3.connect(OUTPUT_DB_PATH)
cursor = conn.cursor()

# Enable SQLite Foreign Key Enforcement
cursor.execute("PRAGMA foreign_keys = ON;")

print("=" * 80)
print("🚀 STARTING FIXED STATS NZ LABOUR MARKET ETL PIPELINE")
print("=" * 80)



🚀 STARTING FIXED STATS NZ LABOUR MARKET ETL PIPELINE


In [3]:
# ------------------------------------------------------------------------------
# STEP 2: HELPER FUNCTIONS FOR DATA CLEANING
# ------------------------------------------------------------------------------
def parse_stats_nz_period(period_val):
    if pd.isna(period_val):
        return None
    try:
        period_str = f"{float(period_val):.2f}"
        year_str, month_str = period_str.split('.')
        year = int(year_str)
        month_or_qtr = int(month_str)

        if month_or_qtr == 3:
            return f"{year}-03-31"
        elif month_or_qtr == 6:
            return f"{year}-06-30"
        elif month_or_qtr == 9:
            return f"{year}-09-30"
        elif month_or_qtr == 12:
            return f"{year}-12-31"
        elif 1 <= month_or_qtr <= 12:
            last_days = {1:31, 2:28, 3:31, 4:30, 5:31, 6:30, 7:31, 8:31, 9:30, 10:31, 11:30, 12:31}
            day = 29 if (month_or_qtr == 2 and (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0))) else last_days[month_or_qtr]
            return f"{year}-{month_or_qtr:02d}-{day:02d}"
        else:
            return None
    except Exception:
        return None

def clean_numeric_value(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().upper()
    if s in ['..', 'N/A', 'S', 'C', 'CONFIDENTIAL', 'SUPPRESSED', '']:
        return np.nan
    try:
        return float(s.replace(',', ''))
    except ValueError:
        return np.nan

def clean_text_value(val, default_text="Unknown"):
    if pd.isna(val):
        return default_text
    s = str(val).strip()
    if s.upper() in ['..', 'N/A', 'S', 'C', 'CONFIDENTIAL', 'SUPPRESSED', 'NONE', '']:
        return default_text
    return s

# ------------------------------------------------------------------------------
# STEP 3: EXTRACT & PRE-CLEAN ALL CSV FILES (MEMORY-OPTIMIZED WITH CHUNKING)
# ------------------------------------------------------------------------------
print("\n[PHASE 1] Extracting & Pre-cleaning Source Data (Memory Optimized)...")
staged_dfs = []

for dataset_code, path in CSV_FILES.items():
    if not os.path.exists(path):
        print(f" ❌ Skipping missing file: {path}")
        continue

    print(f" --> Ingesting {dataset_code}: {os.path.basename(path)}")
    chunk_list = []

    try:
        for chunk in pd.read_csv(path, chunksize=100000, low_memory=False, dtype=str):
            chunk.columns = [c.strip().upper() for c in chunk.columns]

            if 'PERIOD' not in chunk.columns or 'DATA_VALUE' not in chunk.columns:
                continue

            core_dataset_code = dataset_code.split('_')[0]

            # Clean numeric value column
            chunk['CLEAN_VALUE'] = chunk['DATA_VALUE'].apply(clean_numeric_value)
            chunk = chunk.dropna(subset=['CLEAN_VALUE']).copy()

            # Clean period/date
            chunk['DATE_STR'] = chunk['PERIOD'].apply(parse_stats_nz_period)
            chunk = chunk.dropna(subset=['DATE_STR']).copy()
            chunk['DATEKEY'] = chunk['DATE_STR'].str.replace('-', '').astype(int)

            # Standardize attributes
            chunk['SERIES_REF'] = chunk['SERIES_REFERENCE'].apply(lambda x: clean_text_value(x, "UNKNOWN_SERIES")) if 'SERIES_REFERENCE' in chunk.columns else f"{core_dataset_code}_SERIES"
            chunk['SUBJECT_CLEAN'] = chunk['SUBJECT'].apply(lambda x: clean_text_value(x, "General Subject")) if 'SUBJECT' in chunk.columns else "General Subject"
            chunk['GROUP_CLEAN'] = chunk['GROUP'].apply(lambda x: clean_text_value(x, "General Group")) if 'GROUP' in chunk.columns else "General Group"
            chunk['TITLE_1_CLEAN'] = chunk['SERIES_TITLE_1'].apply(lambda x: clean_text_value(x, "Unspecified")) if 'SERIES_TITLE_1' in chunk.columns else "Unspecified"
            chunk['TITLE_2_CLEAN'] = chunk['SERIES_TITLE_2'].apply(lambda x: clean_text_value(x, "Unspecified")) if 'SERIES_TITLE_2' in chunk.columns else "Unspecified"
            chunk['UNITS_CLEAN'] = chunk['UNITS'].apply(lambda x: clean_text_value(x, "Units")) if 'UNITS' in chunk.columns else "Units"
            chunk['STATUS_CLEAN'] = chunk['STATUS'].apply(lambda x: clean_text_value(x, "FINAL")) if 'STATUS' in chunk.columns else "FINAL"
            chunk['DATASET_CODE'] = core_dataset_code

            # Select clean columns
            clean_chunk = chunk[[
                'DATEKEY', 'DATASET_CODE', 'SERIES_REF', 'SUBJECT_CLEAN',
                'GROUP_CLEAN', 'TITLE_1_CLEAN', 'TITLE_2_CLEAN',
                'UNITS_CLEAN', 'STATUS_CLEAN', 'CLEAN_VALUE'
            ]]
            chunk_list.append(clean_chunk)

        if chunk_list:
            df_dataset = pd.concat(chunk_list, ignore_index=True)
            staged_dfs.append(df_dataset)
            print(f" ✓ Successfully processed {len(df_dataset):,} rows for {dataset_code}")

    except Exception as e:
        print(f" ❌ Error processing {dataset_code}: {e}")

df_all = pd.concat(staged_dfs, ignore_index=True)




[PHASE 1] Extracting & Pre-cleaning Source Data (Memory Optimized)...
 --> Ingesting MEI: employment-indicators-july-2026-csv-tables.csv
 ✓ Successfully processed 34,532 rows for MEI
 --> Ingesting HLFS: hlfs-jun26qtr-csv.csv
 ✓ Successfully processed 1,207,728 rows for HLFS
 --> Ingesting LCI: lci-jun26qtr-csv.csv
 ✓ Successfully processed 32,926 rows for LCI
 --> Ingesting LMS: lms-jun26qtr-tables.csv
 ✓ Successfully processed 1,438,668 rows for LMS
 --> Ingesting QES: qes-jun26qtr-csv.csv
 ✓ Successfully processed 198,014 rows for QES


In [4]:
# ------------------------------------------------------------------------------
# STEP 4: CREATE SQL TABLES WITH EXPLICIT DDL & DATA TYPES
# ------------------------------------------------------------------------------
print("\n[PHASE 2] Initializing Explicit Database DDL Schemas...")

# 1. DimDate DDL
cursor.execute("""
CREATE TABLE DimDate (
    DateKey INTEGER PRIMARY KEY,
    FullDate DATE NOT NULL,
    Year TEXT NOT NULL,
    Quarter TEXT NOT NULL,
    QuarterName TEXT NOT NULL,
    Month INTEGER NOT NULL,
    MonthName TEXT NOT NULL
);
""")

# 2. DimDataset DDL
cursor.execute("""
CREATE TABLE DimDataset (
    DatasetKey INTEGER PRIMARY KEY,
    DatasetCode TEXT NOT NULL UNIQUE,
    DatasetName TEXT NOT NULL
);
""")

# 3. DimSeries DDL
cursor.execute("""
CREATE TABLE DimSeries (
    SeriesKey INTEGER PRIMARY KEY,
    SeriesReference TEXT NOT NULL,
    Title1 TEXT NOT NULL,
    Title2 TEXT NOT NULL
);
""")

# 4. DimGroup DDL
cursor.execute("""
CREATE TABLE DimGroup (
    GroupKey INTEGER PRIMARY KEY,
    Subject TEXT NOT NULL,
    "Group" TEXT NOT NULL
);
""")

# 5. DimUnitStatus DDL
cursor.execute("""
CREATE TABLE DimUnitStatus (
    UnitStatusKey INTEGER PRIMARY KEY,
    Units TEXT NOT NULL,
    Status TEXT NOT NULL
);
""")

# 6. FactLabourMarket DDL
cursor.execute("""
CREATE TABLE FactLabourMarket (
    FactID INTEGER PRIMARY KEY AUTOINCREMENT,
    DateKey INTEGER NOT NULL,
    DatasetKey INTEGER NOT NULL,
    SeriesKey INTEGER NOT NULL,
    GroupKey INTEGER NOT NULL,
    UnitStatusKey INTEGER NOT NULL,
    DataValue REAL NOT NULL,
    FOREIGN KEY (DateKey) REFERENCES DimDate(DateKey),
    FOREIGN KEY (DatasetKey) REFERENCES DimDataset(DatasetKey),
    FOREIGN KEY (SeriesKey) REFERENCES DimSeries(SeriesKey),
    FOREIGN KEY (GroupKey) REFERENCES DimGroup(GroupKey),
    FOREIGN KEY (UnitStatusKey) REFERENCES DimUnitStatus(UnitStatusKey)
);
""")

conn.commit()

# ------------------------------------------------------------------------------
# STEP 5: POPULATE DIMENSION TABLES
# ------------------------------------------------------------------------------
print("\n[PHASE 3] Populating Dimension Tables with Formatted Data...")

# 1. DimDate Data Preparation
dates = pd.date_range(start="1980-01-01", end="2030-12-31", freq="D")
dim_date = pd.DataFrame({"FullDate": dates})
dim_date["DateKey"] = dim_date["FullDate"].dt.strftime("%Y%m%d").astype(int)
dim_date["Year"] = dim_date["FullDate"].dt.year.astype(str)  # Cast Year to TEXT for Power BI
dim_date["Quarter"] = "Q" + dim_date["FullDate"].dt.quarter.astype(str)  # Formatted as Q1, Q2, etc.
dim_date["QuarterName"] = dim_date["Quarter"] + " " + dim_date["Year"]
dim_date["Month"] = dim_date["FullDate"].dt.month
dim_date["MonthName"] = dim_date["FullDate"].dt.strftime("%B")
dim_date["FullDate"] = dim_date["FullDate"].dt.strftime("%Y-%m-%d")

dim_date.to_sql("DimDate", conn, if_exists="append", index=False)

# 2. DimDataset Data Preparation
dim_dataset = pd.DataFrame([
    {"DatasetKey": 1, "DatasetCode": "MEI", "DatasetName": "Monthly Employment Indicators"},
    {"DatasetKey": 2, "DatasetCode": "HLFS", "DatasetName": "Household Labour Force Survey"},
    {"DatasetKey": 3, "DatasetCode": "LCI", "DatasetName": "Labour Cost Index"},
    {"DatasetKey": 4, "DatasetCode": "QES", "DatasetName": "Quarterly Employment Survey"},
    {"DatasetKey": 5, "DatasetCode": "LMS", "DatasetName": "Labour Market Statistics Overview"}
])
dim_dataset.to_sql("DimDataset", conn, if_exists="append", index=False)

# 3. DimSeries Data Preparation
dim_series = df_all[['SERIES_REF', 'TITLE_1_CLEAN', 'TITLE_2_CLEAN']].drop_duplicates().reset_index(drop=True)
dim_series.rename(columns={'SERIES_REF': 'SeriesReference', 'TITLE_1_CLEAN': 'Title1', 'TITLE_2_CLEAN': 'Title2'}, inplace=True)
dim_series['SeriesKey'] = range(1, len(dim_series) + 1)
dim_series.to_sql("DimSeries", conn, if_exists="append", index=False)

# 4. DimGroup Data Preparation
dim_group = df_all[['SUBJECT_CLEAN', 'GROUP_CLEAN']].drop_duplicates().reset_index(drop=True)
dim_group.rename(columns={'SUBJECT_CLEAN': 'Subject', 'GROUP_CLEAN': 'Group'}, inplace=True)
dim_group['GroupKey'] = range(1, len(dim_group) + 1)
dim_group.to_sql("DimGroup", conn, if_exists="append", index=False)

# 5. DimUnitStatus Data Preparation
dim_unit_status = df_all[['UNITS_CLEAN', 'STATUS_CLEAN']].drop_duplicates().reset_index(drop=True)
dim_unit_status.rename(columns={'UNITS_CLEAN': 'Units', 'STATUS_CLEAN': 'Status'}, inplace=True)
dim_unit_status['UnitStatusKey'] = range(1, len(dim_unit_status) + 1)
dim_unit_status.to_sql("DimUnitStatus", conn, if_exists="append", index=False)

print(" ✓ All 5 Dimension Tables successfully populated.")




[PHASE 2] Initializing Explicit Database DDL Schemas...

[PHASE 3] Populating Dimension Tables with Formatted Data...
 ✓ All 5 Dimension Tables successfully populated.


In [5]:
# ------------------------------------------------------------------------------
# STEP 6: POPULATE FACT TABLE WITH FOREIGN KEYS
# ------------------------------------------------------------------------------
print("\n[PHASE 4] Building and Populating Fact Table...")

df_fact = df_all.merge(dim_dataset, left_on='DATASET_CODE', right_on='DatasetCode', how='left')
df_fact = df_fact.merge(dim_series, left_on=['SERIES_REF', 'TITLE_1_CLEAN', 'TITLE_2_CLEAN'], right_on=['SeriesReference', 'Title1', 'Title2'], how='left')
df_fact = df_fact.merge(dim_group, left_on=['SUBJECT_CLEAN', 'GROUP_CLEAN'], right_on=['Subject', 'Group'], how='left')
df_fact = df_fact.merge(dim_unit_status, left_on=['UNITS_CLEAN', 'STATUS_CLEAN'], right_on=['Units', 'Status'], how='left')

fact_final = pd.DataFrame({
    'DateKey': df_fact['DATEKEY'],
    'DatasetKey': df_fact['DatasetKey'],
    'SeriesKey': df_fact['SeriesKey'],
    'GroupKey': df_fact['GroupKey'],
    'UnitStatusKey': df_fact['UnitStatusKey'],
    'DataValue': df_fact['CLEAN_VALUE'].astype(float)
})

fact_final.to_sql("FactLabourMarket", conn, if_exists="append", index=False)
print(f" ✓ FactLabourMarket successfully populated with {len(fact_final):,} rows.")

# ------------------------------------------------------------------------------
# STEP 7: BUILD AUDIT VIEWS
# ------------------------------------------------------------------------------
print("\n[PHASE 5] Creating Database Audit Views...")

cursor.execute("""
CREATE VIEW v_audit_dataset_summary AS
SELECT 
    d.DatasetCode,
    d.DatasetName,
    COUNT(f.FactID) AS DW_Total_Rows,
    ROUND(SUM(f.DataValue), 2) AS DW_Sum_DataValue,
    MIN(dt.FullDate) AS Earliest_Date,
    MAX(dt.FullDate) AS Latest_Date
FROM FactLabourMarket f
JOIN DimDataset d ON f.DatasetKey = d.DatasetKey
JOIN DimDate dt ON f.DateKey = dt.DateKey
GROUP BY d.DatasetCode, d.DatasetName;
""")

cursor.execute("""
CREATE VIEW v_yearly_labour_trends AS
SELECT 
    dt.Year,
    ds.DatasetCode,
    COUNT(f.FactID) AS Total_Records,
    ROUND(SUM(f.DataValue), 2) AS Total_Value,
    ROUND(AVG(f.DataValue), 2) AS Average_Value
FROM FactLabourMarket f
JOIN DimDate dt ON f.DateKey = dt.DateKey
JOIN DimDataset ds ON f.DatasetKey = ds.DatasetKey
GROUP BY dt.Year, ds.DatasetCode;
""")

conn.commit()
print(" ✓ Views 'v_audit_dataset_summary' and 'v_yearly_labour_trends' successfully created.")

print("=" * 80)


[PHASE 4] Building and Populating Fact Table...
 ✓ FactLabourMarket successfully populated with 2,911,868 rows.

[PHASE 5] Creating Database Audit Views...
 ✓ Views 'v_audit_dataset_summary' and 'v_yearly_labour_trends' successfully created.


In [6]:
# ------------------------------------------------------------------------------
# STEP 8: DATABASE SCHEMA & STAR SCHEMA INTEGRITY AUDIT
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("🔎 STEP 6: DATABASE SCHEMA & STAR SCHEMA INTEGRITY AUDIT")
print("=" * 80)

# 8.1 Table Inventory
tables_df = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(f"\n[AUDIT 8.1] Database Table Inventory (Total Tables: {len(tables_df)}):")
for idx, tbl in enumerate(tables_df['name'], 1):
    row_cnt = pd.read_sql(f"SELECT COUNT(*) FROM {tbl}", conn).squeeze()
    print(f" {idx}. Table Name: '{tbl}' | Total Rows: {row_cnt:,}")

# 8.2 Star Schema Foreign Key Integrity Check
print("\n[AUDIT 8.2] Star Schema Foreign Key Integrity Check:")
orphan_dates = pd.read_sql("SELECT COUNT(*) FROM FactLabourMarket f LEFT JOIN DimDate d ON f.DateKey = d.DateKey WHERE d.DateKey IS NULL", conn).squeeze()
orphan_dataset = pd.read_sql("SELECT COUNT(*) FROM FactLabourMarket f LEFT JOIN DimDataset d ON f.DatasetKey = d.DatasetKey WHERE d.DatasetKey IS NULL", conn).squeeze()
orphan_series = pd.read_sql("SELECT COUNT(*) FROM FactLabourMarket f LEFT JOIN DimSeries d ON f.SeriesKey = d.SeriesKey WHERE d.SeriesKey IS NULL", conn).squeeze()
orphan_group = pd.read_sql("SELECT COUNT(*) FROM FactLabourMarket f LEFT JOIN DimGroup d ON f.GroupKey = d.GroupKey WHERE d.GroupKey IS NULL", conn).squeeze()
orphan_unit = pd.read_sql("SELECT COUNT(*) FROM FactLabourMarket f LEFT JOIN DimUnitStatus d ON f.UnitStatusKey = d.UnitStatusKey WHERE d.UnitStatusKey IS NULL", conn).squeeze()

print(f" • Unmatched DateKey Orphans     : {orphan_dates} (Expected: 0)")
print(f" • Unmatched DatasetKey Orphans  : {orphan_dataset} (Expected: 0)")
print(f" • Unmatched SeriesKey Orphans   : {orphan_series} (Expected: 0)")
print(f" • Unmatched GroupKey Orphans    : {orphan_group} (Expected: 0)")
print(f" • Unmatched UnitStatusKey Orphans: {orphan_unit} (Expected: 0)")

# 8.3 Summary View Output
print("\n[AUDIT 8.3] Audit Dataset Summary (v_audit_dataset_summary):")
audit_summary = pd.read_sql("SELECT * FROM v_audit_dataset_summary", conn)
print(audit_summary.to_string(index=False))

conn.close()

print("\n" + "=" * 80)
print("✅ END-TO-END PIPELINE AND DATABASE RE-BUILD COMPLETE!")



🔎 STEP 6: DATABASE SCHEMA & STAR SCHEMA INTEGRITY AUDIT

[AUDIT 8.1] Database Table Inventory (Total Tables: 7):
 1. Table Name: 'DimDate' | Total Rows: 18,628
 2. Table Name: 'DimDataset' | Total Rows: 5
 3. Table Name: 'DimSeries' | Total Rows: 54,735
 4. Table Name: 'DimGroup' | Total Rows: 146
 5. Table Name: 'DimUnitStatus' | Total Rows: 16
 6. Table Name: 'FactLabourMarket' | Total Rows: 2,911,868
 7. Table Name: 'sqlite_sequence' | Total Rows: 1

[AUDIT 8.2] Star Schema Foreign Key Integrity Check:
 • Unmatched DateKey Orphans     : 0 (Expected: 0)
 • Unmatched DatasetKey Orphans  : 0 (Expected: 0)
 • Unmatched SeriesKey Orphans   : 0 (Expected: 0)
 • Unmatched GroupKey Orphans    : 0 (Expected: 0)
 • Unmatched UnitStatusKey Orphans: 0 (Expected: 0)

[AUDIT 8.3] Audit Dataset Summary (v_audit_dataset_summary):
DatasetCode                       DatasetName  DW_Total_Rows  DW_Sum_DataValue Earliest_Date Latest_Date
       HLFS     Household Labour Force Survey        1207728     